# Fase 2 - mT5-base Answer-Aware Question Generation (T4 × 2)

Notebook ini adalah **Fase 2: retraining dan perbaikan pipeline training** dari eksperimen mT5-small sebelumnya, tetapi desainnya diperbaiki pada titik yang terbukti lemah.

Perubahan utama:

1. **Model** diganti menjadi `google/mt5-base`.
2. **Multi-GPU** menggunakan DDP melalui `accelerate.notebook_launcher`, sehingga dua Tesla T4 bekerja sebagai dua proses terpisah. Tidak memakai DataParallel.
3. **Answer control diperkuat** dengan menandai lokasi answer langsung di dalam context menggunakan token khusus `<answer> ... </answer>`.
4. Prompt juga membawa `answer_type` dan `formula_context` agar model mengetahui jenis fokus yang harus dibentuk menjadi pertanyaan.
5. Context panjang dipotong **berpusat pada lokasi answer**, sehingga span yang menjadi target pertanyaan tidak mudah hilang.
6. Data yang answer span-nya tidak benar-benar terdapat di context dibuang.
7. Data formula sintetis yang terkontrol ditambahkan **hanya ke training**. Validation utama tetap berasal dari IDK-MRC + SQuAD agar hasil dapat dibandingkan dengan baseline sebelumnya.
8. Disediakan formula test terpisah untuk mengukur perilaku pada materi berisi rumus.
9. Setelah training dilakukan **answer-sensitivity audit**: context sama dengan answer berbeda harus menghasilkan pertanyaan yang berubah. Ini menargetkan blind spot terbesar pada run sebelumnya.
10. Evaluasi utama tetap ROUGE-1, ROUGE-2, ROUGE-L, dan BLEU-4, tetapi ditambah diagnostic answer-type alignment dan formula evaluation.

## Batasan yang harus dipahami

Deteksi apakah dokumen memiliki rumus **bukan kemampuan yang otomatis muncul dari fine-tuning QG**. Notebook ini menyediakan detector formula berbasis aturan dan mempertahankan formula di context. Fine-tuning juga diberi contoh formula sintetis agar model mendapat sinyal QG untuk rumus. Untuk deployment dengan PDF/PPT nyata, detector formula tetap sebaiknya menjadi tahap preprocessing tersendiri sebelum QG.

Catatan: penambahan token khusus mengubah ukuran vocabulary dan embedding; model hasil training harus disimpan bersama tokenizer baru.

Jalankan cell secara berurutan di Kaggle dengan **GPU T4 × 2** dan Internet aktif.

## Cell 1 — Instalasi dependensi

Dependensi dipasang dari Python API sehingga notebook tetap 100% Python: tidak ada magic command, shell script, `torchrun`, atau file `.py` eksternal.

Versi Transformers diarahkan ke rilis 5.x yang menyediakan `eval_strategy` dan `processing_class`. Untuk mT5 pada T4, FP16 tetap dimatikan sesuai desain eksperimen sebelumnya.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "transformers>=5.10.0,<5.18.0",
    "datasets>=4.0.0,<5.0.0",
    "accelerate>=1.5.0,<2.0.0",
    "evaluate>=0.4.5,<0.5.0",
    "rouge_score>=0.1.2",
    "sacrebleu>=2.5.0",
    "sentencepiece>=0.2.0",
    "protobuf>=4.25.0",
    "packaging>=24.0",
    "pandas>=2.0.0",
]

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--disable-pip-version-check",
    *PACKAGES,
]

result = subprocess.run(command, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-2000:])
    print(result.stderr[-4000:])
    raise RuntimeError(
        "Instalasi dependensi gagal. Pastikan Internet Kaggle aktif, lalu jalankan ulang cell ini."
    )

print("Dependensi selesai dipasang.")


## Cell 2 — Import, seed, verifikasi T4 × 2, dan konfigurasi global

Ada perbedaan penting dari notebook lama: **proses utama tidak boleh menyentuh CUDA sebelum DDP diluncurkan**. Pada run sebelumnya `torch.manual_seed()` ikut membuat CUDA aktif, sehingga `accelerate.notebook_launcher` terpaksa jatuh ke DataParallel.

Di notebook ini seed PyTorch/CUDA disetel di dalam `train_worker()`, bukan di proses utama. Jumlah GPU dibaca memakai `nvidia-smi` agar verifikasi hardware tidak menginisialisasi CUDA.

In [ ]:
import os
import sys
import subprocess
import random
import re
import json
import math
from collections import Counter, defaultdict

# Environment harus dipasang sebelum import torch untuk membantu fragmentasi allocator.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import pandas as pd
import torch
import datasets
import evaluate
import transformers
from packaging import version
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
MODEL_NAME = "google/mt5-base"

OUTPUT_DIR = "/kaggle/working/mt5-base-aqg-checkpoints"
FINAL_DIR = "/kaggle/working/mt5-base-aqg-final"

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 64
MAX_ANSWER_TOKENS = 64
VAL_RATIO = 0.10

# Dua T4. Batch 2/GPU dan accumulation 4 memberikan effective batch = 16.
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
TARGET_EFFECTIVE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = TARGET_EFFECTIVE_BATCH_SIZE // (PER_DEVICE_TRAIN_BATCH_SIZE * 2)

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LABEL_SMOOTHING = 0.10
NUM_EPOCHS = 3
EVAL_STEPS = 250

# Formula augmentation sengaja dipisahkan dari benchmark umum.
USE_FORMULA_AUGMENTATION = True
FORMULA_TRAIN_PER_LANG = 240
FORMULA_EVAL_PER_LANG = 60

# Tidak memanggil torch.manual_seed/torch.cuda di proses utama.
random.seed(SEED)
np.random.seed(SEED)

# Verifikasi GPU tanpa torch.cuda.*.
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    timeout=30,
)

gpu_lines = [line.strip() for line in smi.stdout.strip().splitlines() if line.strip()] if smi.returncode == 0 else []
NUM_GPUS = len(gpu_lines)

if NUM_GPUS != 2:
    raise RuntimeError(
        f"Notebook ini ditargetkan untuk T4 x2, tetapi terdeteksi {NUM_GPUS} GPU: {gpu_lines}. "
        "Ubah Accelerator Kaggle menjadi 2× Tesla T4."
    )

if version.parse(transformers.__version__) < version.parse("5.10.0"):
    raise RuntimeError(
        f"Transformers {transformers.__version__} terlalu lama. Jalankan ulang Cell 1 lalu restart session."
    )

print("Python          :", sys.version.split()[0])
print("PyTorch         :", torch.__version__)
print("Transformers    :", transformers.__version__)
print("Datasets        :", datasets.__version__)
print("Evaluate        :", evaluate.__version__)
print("GPU             :", " | ".join(gpu_lines))
print("Jumlah GPU      :", NUM_GPUS)
print("Model           :", MODEL_NAME)
print("Batch/GPU       :", PER_DEVICE_TRAIN_BATCH_SIZE)
print("Grad accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch :", PER_DEVICE_TRAIN_BATCH_SIZE * NUM_GPUS * GRADIENT_ACCUMULATION_STEPS)
print("FP16            : False")
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized(), "(harus False)")


## Cell 3 — Parsing dataset, pembersihan answer span, formula detector, dan formula augmentation

Cell ini melakukan empat pekerjaan penting:

- Membaca `rifkiaputri/idk-mrc` dan `squad`.
- Menghapus `is_impossible=True` pada IDK-MRC, membuang answer kosong, dan membuang answer yang tidak ditemukan di context.
- Membagi **berdasarkan context**, bukan baris QA. Ini mencegah paragraf yang sama masuk train dan validation sekaligus.
- Menambahkan dataset sintetis formula **hanya ke train**. Validation umum tetap murni dari data sumber agar benchmark tidak berubah.

`detect_formula_content()` adalah detector awal untuk konteks yang mengandung LaTeX, persamaan, operator matematika, atau pola persamaan sederhana. Detector ini nanti bisa dipakai lagi pada pipeline PDF/PPT.

In [ ]:
import time
from tqdm.auto import tqdm

# Regex formula diamankan dari backtracking tak terbatas
FORMULA_PATTERNS = [
    re.compile(r"\$\$[\s\S]*?\$\$", re.DOTALL),
    re.compile(r"\$(?!\$)[^$\n]{1,180}\$(?!\$)"),
    re.compile(r"\\\([^\n]{1,180}?\\\)"),
    re.compile(r"\\\[[\s\S]{1,240}?\\\]"),
    re.compile(r"\\(?:frac|dfrac|tfrac|sqrt|sum|int|prod|lim|partial|nabla|alpha|beta|gamma|theta|lambda|pi|sigma|mu|rho)\b(?:\s*\{[^}]*\})?"),
    re.compile(r"(?<!\w)(?:[A-Za-zα-ωΑ-Ω][A-Za-z0-9_{}'^]*|[A-Za-zα-ωΑ-Ω]\s*\([^)]{0,20}\)|d\s*\([^)]{0,35}\)\s*/\s*dx)\s*(?:=|≈|≤|≥|≠)\s*[^.;,\n]{1,100}"),
    re.compile(r"(?<!\w)(?:d/dx|d\s*\([^)]*\)/dx|∫|Σ|∑|sqrt\s*\()[^.;,\n]{0,100}"),
    re.compile(r"(?<!\w)\d+(?:[.,]\d+)?\s*(?:\+|-|\*|/|×|÷|\^|<=|>=|!=)\s*\d+(?:[.,]\d+)?"),
]


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def detect_formula_spans(text):
    text = str(text)
    spans = []
    for pattern in FORMULA_PATTERNS:
        for match in pattern.finditer(text):
            if match.end() > match.start():
                spans.append((match.start(), match.end(), text[match.start():match.end()]))
    spans.sort(key=lambda item: (item[0], -(item[1] - item[0])))
    reduced = []
    for start, end, _ in spans:
        if not reduced or start >= reduced[-1][1]:
            reduced.append((start, end, text[start:end]))
        else:
            old_start, old_end, _ = reduced[-1]
            reduced[-1] = (old_start, max(old_end, end), text[old_start:max(old_end, end)])
    return reduced[:30]


def detect_formula_content(text):
    spans = detect_formula_spans(text)
    return bool(spans), [span[2] for span in spans]


def convert_to_slide_style(context, answer=None):
    """Ubah prosa menjadi format ringkas yang menyerupai isi slide PPT."""
    text = normalize_space(context)
    if not text:
        return text
    sentences = [x.strip(" -•\t") for x in re.split(r"(?<=[.!?])\s+|\s*[•▪]\s*", text) if x.strip(" -•\t")]
    if not sentences:
        sentences = [text]
    topic = sentences[0].rstrip(". ")
    bullets = sentences[1:] or sentences[:1]
    # Jangan kehilangan answer yang sedang menjadi target QA saat parafrasa struktur.
    if answer and not any(str(answer).casefold() in part.casefold() for part in sentences):
        bullets.append(str(answer))
    return "Topik: " + topic + " | Poin Penting: " + " ".join("- " + part.rstrip(". ") + "." for part in bullets)


def _format_synthetic_value(value):
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value)


def _make_additional_formula_spec(name_id, name_en, formula, variants, calc, body_id, body_en, unit="", unit_en=None):
    """Factory keeps each synthetic formula template consistent across both languages."""
    unit_en = unit if unit_en is None else unit_en

    def _render(template, a, b, result):
        text = template.replace("{formula}", formula).replace("{a}", _format_synthetic_value(a))
        text = text.replace("{b}", _format_synthetic_value(b)).replace("{result}", _format_synthetic_value(result))
        text = text.replace("{unit}", unit).replace("  .", ".").replace(" .", ".")
        return normalize_space(text)

    def build(a, b):
        result = calc(a, b)
        return _render(body_id, a, b, result), f"{_format_synthetic_value(result)}{(' ' + unit) if unit else ''}"

    def build_en(a, b):
        result = calc(a, b)
        return _render(body_en, a, b, result), f"{_format_synthetic_value(result)}{(' ' + unit_en) if unit_en else ''}"

    return {
        "name_id": name_id, "name_en": name_en, "formula": formula,
        "variants": variants, "build": build, "build_en": build_en,
        "q_formula_id": [
            f"Apa rumus {name_id}?", f"Persamaan apa yang digunakan pada {name_id}?",
            f"Bagaimana bentuk rumus {name_id}?", f"Rumus apa yang menyatakan {name_id}?",
        ],
        "q_formula_en": [
            f"What is the formula for {name_en}?", f"Which equation is used for {name_en}?",
            f"How is the {name_en} formula written?", f"Which formula expresses {name_en}?",
        ],
        "q_num_id": [
            f"Berapa hasil perhitungan {name_id} pada contoh tersebut?",
            f"Berapa nilai yang diperoleh dari {name_id}?",
            f"Berapakah hasil numerik {name_id} berdasarkan data tersebut?",
            f"Berapa nilai hasil perhitungannya pada contoh {name_id}?",
        ],
        "q_num_en": [
            f"What is the calculated result for {name_en} in this example?",
            f"What value is obtained from {name_en}?",
            f"What is the numerical result for {name_en}?",
            f"What value results from this {name_en} example?",
        ],
    }


def answer_type(answer):
    """
    Validasi cepat tanpa risiko Catastrophic Backtracking (ReDoS).
    """
    value = normalize_space(answer)
    if not value:
        return "text"

    # 1. Cek tanggal
    if re.fullmatch(r"(?:0?[1-9]|[12][0-9]|3[01])[-/.](?:0?[1-9]|1[0-2])[-/.](?:19|20)\d{2}", value) or re.fullmatch(r"(?:19|20)\d{2}", value):
        return "date"

    # 2. Cek angka / kuantitas besaran (Python Native: instan dalam mikrodetik)
    words = value.split()
    if words:
        clean_num = words[0].replace(",", ".").rstrip("%°")
        try:
            float(clean_num)
            # Jika kata pertama angka dan panjang jawaban maksimal 3 kata (angka + satuan seperti "40 cm²")
            if len(words) <= 3:
                return "number"
        except ValueError:
            pass

    # 3. Cek formula
    has_formula, _ = detect_formula_content(value)
    if has_formula:
        return "formula"

    return "text"


def as_records(obj):
    if obj is None:
        return []
    if isinstance(obj, dict):
        keys = list(obj.keys())
        if not keys:
            return []
        first = obj[keys[0]]
        if isinstance(first, (list, tuple, np.ndarray)):
            n = len(first)
            return [{key: obj[key][i] for key in keys} for i in range(n)]
        return [obj]
    if isinstance(obj, (list, tuple, np.ndarray)):
        return list(obj)
    return []


def to_int(value, default=-1):
    try:
        if isinstance(value, (list, tuple, np.ndarray)):
            value = value[0] if len(value) else default
        return int(value)
    except (TypeError, ValueError):
        return default


def to_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    return bool(value)


def first_answer(answers):
    for record in as_records(answers):
        if not isinstance(record, dict):
            continue
        text = record.get("text")
        if isinstance(text, (list, tuple, np.ndarray)):
            text = text[0] if len(text) else None
        if text is None or not str(text).strip():
            continue
        return str(text), to_int(record.get("answer_start", -1))
    return None, -1


def parse_idk_mrc(dataset_obj):
    stats = Counter()
    rows = []
    seen = set()
    formula_cache = {}

    for split_name in dataset_obj.keys():
        contexts_list = dataset_obj[split_name]
        for example in tqdm(contexts_list, desc=f"Parsing IDK-MRC ({split_name})"):
            context = example.get("context")
            if not context:
                continue
            context = str(context).strip()

            if context not in formula_cache:
                formula_cache[context] = detect_formula_content(context)
            has_formula, formula_spans = formula_cache[context]

            for qa in as_records(example.get("qas")):
                stats["total_qas"] += 1
                if not isinstance(qa, dict):
                    continue
                if to_bool(qa.get("is_impossible", False)):
                    stats["impossible_removed"] += 1
                    continue

                raw_ans, start = first_answer(qa.get("answers"))
                if raw_ans is None:
                    stats["no_answer_removed"] += 1
                    continue

                raw_q = qa.get("question")
                if raw_q is None:
                    stats["empty_field"] += 1
                    continue

                clean_ans = normalize_space(raw_ans)
                clean_q = normalize_space(raw_q)
                if not clean_ans or not clean_q:
                    stats["empty_field"] += 1
                    continue

                start = int(start)
                if start < 0 or context[start:start + len(clean_ans)] != clean_ans:
                    start = context.find(clean_ans)
                if start < 0:
                    start_lower = context.lower().find(clean_ans.lower())
                    if start_lower >= 0:
                        start = start_lower
                        clean_ans = context[start:start + len(clean_ans)]
                    else:
                        stats["answer_not_in_context"] += 1
                        continue

                row = {
                    "lang": "id",
                    "context": context,
                    "answer": clean_ans,
                    "question": clean_q,
                    "answer_start": start,
                    "answer_type": answer_type(clean_ans),
                    "has_formula": has_formula,
                    "formula_spans": json.dumps(formula_spans, ensure_ascii=False),
                    "source": "idk_mrc",
                }
                key = (row["context"], row["question"], row["answer"])
                if key in seen:
                    stats["duplicate_removed"] += 1
                    continue
                seen.add(key)
                rows.append(row)

    stats["unique_contexts"] = len(formula_cache)
    stats["kept"] = len(rows)
    return rows, stats


def parse_squad(dataset_obj):
    stats = Counter()
    rows = []
    seen = set()
    formula_cache = {}

    for example in tqdm(dataset_obj, desc="Parsing SQuAD (en)"):
        raw_ans, start = first_answer(example.get("answers"))
        if raw_ans is None:
            stats["no_answer_removed"] += 1
            continue

        ctx = str(example.get("context", "")).strip()
        if not ctx:
            stats["empty_field"] += 1
            continue

        if ctx not in formula_cache:
            formula_cache[ctx] = detect_formula_content(ctx)
        has_formula, formula_spans = formula_cache[ctx]

        clean_ans = normalize_space(raw_ans)
        clean_q = normalize_space(example.get("question", ""))
        if not clean_ans or not clean_q:
            stats["empty_field"] += 1
            continue

        start = int(start)
        if start < 0 or ctx[start:start + len(clean_ans)] != clean_ans:
            start = ctx.find(clean_ans)
        if start < 0:
            start_lower = ctx.lower().find(clean_ans.lower())
            if start_lower >= 0:
                start = start_lower
                clean_ans = ctx[start:start + len(clean_ans)]
            else:
                stats["answer_not_in_context"] += 1
                continue

        row = {
            "lang": "en",
            "context": ctx,
            "answer": clean_ans,
            "question": clean_q,
            "answer_start": start,
            "answer_type": answer_type(clean_ans),
            "has_formula": has_formula,
            "formula_spans": json.dumps(formula_spans, ensure_ascii=False),
            "source": "squad",
        }
        key = (row["context"], row["question"], row["answer"])
        if key in seen:
            stats["duplicate_removed"] += 1
            continue
        seen.add(key)
        rows.append(row)

    stats["kept"] = len(rows)
    return rows, stats


def load_dataset_safe(repo_id, **kwargs):
    errors = []
    attempts = [kwargs, {**kwargs, "revision": "refs/convert/parquet"}]
    for attempt_idx, attempt in enumerate(attempts, start=1):
        print(f"    [LOAD {attempt_idx}/{len(attempts)}] Mencoba {repo_id} dengan {attempt}", flush=True)
        start_time = time.time()
        try:
            dataset = load_dataset(repo_id, **attempt)
            print(f"    [LOAD OK] {repo_id} berhasil dalam {time.time() - start_time:.1f} detik.", flush=True)
            return dataset
        except Exception as exc:
            elapsed = time.time() - start_time
            errors.append(f"{repo_id} {attempt}: {type(exc).__name__}: {str(exc)[:500]}")
            print(f"    [LOAD FAIL] Percobaan {attempt_idx} gagal ({elapsed:.1f}s): {str(exc)[:150]}", flush=True)
    raise RuntimeError("Gagal memuat dataset.\n" + "\n".join(errors))


def split_by_context(records, val_ratio, seed):
    groups = defaultdict(list)
    for row in records:
        groups[row["context"]].append(row)
    contexts = list(groups)
    rng = random.Random(seed)
    rng.shuffle(contexts)

    target_val_rows = int(round(len(records) * val_ratio))
    train, val = [], []
    for context in contexts:
        if len(val) < target_val_rows:
            val.extend(groups[context])
        else:
            train.extend(groups[context])
    return train, val


# ----------------------------- 1. Load source data -----------------------------
print("\n" + "=" * 78, flush=True)
print("[1/8] MEMUAT DATASET DARI HUGGING FACE HUB", flush=True)
print("=" * 78, flush=True)

raw_idk = load_dataset_safe("rifkiaputri/idk-mrc")
if not isinstance(raw_idk, dict):
    raw_idk = {"train": raw_idk}

raw_squad = load_dataset_safe("rajpurkar/squad", split="train")

print("\nSchema dataset:", flush=True)
for split_name in raw_idk.keys():
    print(f"  IDK-MRC {split_name}: {len(raw_idk[split_name]):,} contexts, cols={raw_idk[split_name].column_names}", flush=True)
print(f"  SQuAD train: {len(raw_squad):,} rows, cols={raw_squad.column_names}", flush=True)

# ----------------------------- 2. Parsing -----------------------------
print("\n" + "=" * 78, flush=True)
print("[2/8] PARSING DATASET", flush=True)
print("=" * 78, flush=True)

parse_start = time.time()
idk_rows, idk_stats = parse_idk_mrc(raw_idk)
print(f"[2A OK] {len(idk_rows):,} QA IDK-MRC valid ({time.time() - parse_start:.1f}s)", flush=True)

parse_start = time.time()
squad_rows, squad_stats = parse_squad(raw_squad)
print(f"[2B OK] {len(squad_rows):,} QA SQuAD valid ({time.time() - parse_start:.1f}s)", flush=True)

# ----------------------------- 3. Balance 1:1 -----------------------------
print("\n" + "=" * 78, flush=True)
print("[3/8] BALANCING 1:1 ANTARA INDONESIA DAN INGGRIS", flush=True)
print("=" * 78, flush=True)

rng = random.Random(SEED)
n_per_lang = len(idk_rows)
idk_selected = idk_rows.copy()
squad_selected = rng.sample(squad_rows, n_per_lang)
rng.shuffle(idk_selected)
rng.shuffle(squad_selected)

all_general = idk_selected + squad_selected
rng.shuffle(all_general)

print(f"IDK-MRC dipakai : {len(idk_selected):,}", flush=True)
print(f"SQuAD dipakai   : {len(squad_selected):,}", flush=True)
print(f"Total general   : {len(all_general):,}", flush=True)

# ----------------------------- 4. Split by context -----------------------------
print("\n" + "=" * 78, flush=True)
print("[4/8] TRAIN / VALIDATION SPLIT 90:10 BERDASARKAN CONTEXT", flush=True)
print("=" * 78, flush=True)

train_general, val_general = [], []
for lang in ("id", "en"):
    lang_rows = [row for row in all_general if row["lang"] == lang]
    lang_train, lang_val = split_by_context(lang_rows, VAL_RATIO, SEED)
    train_general.extend(lang_train)
    val_general.extend(lang_val)
    print(f"{lang}: total={len(lang_rows):,}, train={len(lang_train):,}, val={len(lang_val):,}", flush=True)

# ----------------------------- 5. Synthetic formula data -----------------------------
FORMULA_SPECS = [
    {
        "name_id": "luas persegi panjang", "name_en": "rectangle area",
        "formula": "A = p × l",
        "variants": [(8, 5), (9, 4), (12, 6), (15, 3)],
        "build": lambda a, b: (f"Luas persegi panjang dihitung dengan rumus A = p × l. Jika p = {a} cm dan l = {b} cm, maka luas persegi panjang adalah {a*b} cm².", f"{a*b} cm²"),
        "build_en": lambda a, b: (f"The area of a rectangle is calculated using A = p × l. If p = {a} cm and l = {b} cm, the rectangle area is {a*b} cm².", f"{a*b} cm²"),
        "q_formula_id": ["Apa rumus yang digunakan untuk menghitung luas persegi panjang?", "Bagaimana rumus luas persegi panjang?", "Persamaan apa yang digunakan untuk mencari luas persegi panjang?", "Rumus apa yang digunakan pada perhitungan luas persegi panjang?"],
        "q_formula_en": ["What formula is used to calculate the area of a rectangle?", "What is the formula for the area of a rectangle?", "Which equation is used to find the rectangle area?", "What equation is used for calculating rectangle area?"],
        "q_num_id": ["Berapa luas persegi panjang tersebut?", "Berapa nilai luas persegi panjang?", "Berapa hasil perhitungan luas persegi panjang?", "Berapakah luas persegi panjang pada kasus tersebut?"],
        "q_num_en": ["What is the area of the rectangle?", "How large is the rectangle's area?", "What is the calculated rectangle area?", "How much is the rectangle area?"],
    },
    {
        "name_id": "luas segitiga", "name_en": "triangle area",
        "formula": "A = 1/2 × a × t",
        "variants": [(10, 6), (12, 5), (14, 8), (16, 7)],
        "build": lambda a, b: (f"Rumus luas segitiga adalah A = 1/2 × a × t. Jika alas a = {a} cm dan tinggi t = {b} cm, maka luas segitiga adalah {a*b/2:g} cm².", f"{a*b/2:g} cm²"),
        "build_en": lambda a, b: (f"The triangle area formula is A = 1/2 × a × t. If base a = {a} cm and height t = {b} cm, the triangle area is {a*b/2:g} cm².", f"{a*b/2:g} cm²"),
        "q_formula_id": ["Apa rumus luas segitiga?", "Persamaan apa yang digunakan untuk menghitung luas segitiga?", "Bagaimana cara menuliskan rumus luas segitiga?", "Rumus apa yang digunakan untuk mencari luas segitiga?"],
        "q_formula_en": ["What is the formula for the area of a triangle?", "Which equation calculates the triangle area?", "How is the triangle area formula written?", "What formula is used to find the triangle area?"],
        "q_num_id": ["Berapa luas segitiga tersebut?", "Berapa hasil luas segitiga?", "Berapakah nilai luas segitiga?", "Berapa luas segitiga pada kondisi tersebut?"],
        "q_num_en": ["What is the area of the triangle?", "What is the calculated triangle area?", "How much is the triangle area?", "What value do we obtain for the triangle area?"],
    },
    {
        "name_id": "teorema Pythagoras", "name_en": "Pythagorean theorem",
        "formula": "c² = a² + b²",
        "variants": [(3, 4), (6, 8), (5, 12), (9, 12)],
        "build": lambda a, b: (f"Teorema Pythagoras dinyatakan dengan c² = a² + b². Jika a = {a} cm dan b = {b} cm, panjang sisi miring adalah {math.hypot(a,b):g} cm.", f"{math.hypot(a,b):g} cm"),
        "build_en": lambda a, b: (f"The Pythagorean theorem is expressed as c² = a² + b². If a = {a} cm and b = {b} cm, the hypotenuse length is {math.hypot(a,b):g} cm.", f"{math.hypot(a,b):g} cm"),
        "q_formula_id": ["Apa rumus Teorema Pythagoras?", "Persamaan apa yang menyatakan Teorema Pythagoras?", "Bagaimana bentuk rumus Pythagoras?", "Rumus apa yang digunakan pada Teorema Pythagoras?"],
        "q_formula_en": ["What is the Pythagorean theorem formula?", "Which equation expresses the Pythagorean theorem?", "How is the Pythagorean formula written?", "What equation is used for the Pythagorean theorem?"],
        "q_num_id": ["Berapa panjang sisi miringnya?", "Berapa nilai sisi miring segitiga tersebut?", "Berapa hasil panjang sisi miring?", "Berapa panjang sisi c pada kasus tersebut?"],
        "q_num_en": ["What is the hypotenuse length?", "What is the value of the hypotenuse?", "What is the calculated hypotenuse length?", "How long is side c in this case?"],
    },
    {
        "name_id": "massa jenis", "name_en": "density",
        "formula": "ρ = m/V",
        "variants": [(200, 50), (300, 75), (480, 120), (600, 150)],
        "build": lambda a, b: (f"Massa jenis dihitung dengan rumus ρ = m/V. Jika massa m = {a} g dan volume V = {b} cm³, massa jenisnya adalah {a/b:g} g/cm³.", f"{a/b:g} g/cm³"),
        "build_en": lambda a, b: (f"Density is calculated using ρ = m/V. If mass m = {a} g and volume V = {b} cm³, the density is {a/b:g} g/cm³.", f"{a/b:g} g/cm³"),
        "q_formula_id": ["Apa rumus massa jenis?", "Persamaan apa yang digunakan untuk menghitung massa jenis?", "Bagaimana rumus massa jenis dituliskan?", "Rumus apa yang digunakan untuk mencari massa jenis?"],
        "q_formula_en": ["What is the density formula?", "Which equation calculates density?", "How is the density formula written?", "What formula is used to find density?"],
        "q_num_id": ["Berapa massa jenis benda tersebut?", "Berapa hasil massa jenisnya?", "Berapa nilai massa jenis berdasarkan data tersebut?", "Berapakah massa jenis pada kasus tersebut?"],
        "q_num_en": ["What is the density of the object?", "What is the calculated density?", "What density value is obtained from the data?", "What is the density in this case?"],
    },
    {
        "name_id": "kecepatan", "name_en": "speed",
        "formula": "v = s/t",
        "variants": [(120, 2), (150, 3), (240, 4), (360, 6)],
        "build": lambda a, b: (f"Kecepatan dirumuskan sebagai v = s/t. Sebuah benda menempuh jarak s = {a} m selama t = {b} s, sehingga kecepatannya adalah {a/b:g} m/s.", f"{a/b:g} m/s"),
        "build_en": lambda a, b: (f"Speed is defined by v = s/t. An object travels s = {a} m in t = {b} s, so its speed is {a/b:g} m/s.", f"{a/b:g} m/s"),
        "q_formula_id": ["Apa rumus kecepatan?", "Persamaan apa yang digunakan untuk menghitung kecepatan?", "Bagaimana rumus kecepatan dituliskan?", "Rumus apa yang digunakan untuk menentukan kecepatan?"],
        "q_formula_en": ["What is the speed formula?", "Which equation calculates speed?", "How is the speed formula written?", "What formula determines speed?"],
        "q_num_id": ["Berapa kecepatan benda tersebut?", "Berapa nilai kecepatannya?", "Berapa hasil perhitungan kecepatan?", "Berapakah kecepatan pada kondisi tersebut?"],
        "q_num_en": ["What is the object's speed?", "What is the speed value?", "What is the calculated speed?", "What speed is obtained in this case?"],
    },
    {
        "name_id": "Hukum Ohm", "name_en": "Ohm's law",
        "formula": "V = I × R",
        "variants": [(2, 6), (3, 4), (5, 8), (1.5, 12)],
        "build": lambda a, b: (f"Hukum Ohm dinyatakan dengan V = I × R. Jika arus I = {a} A dan hambatan R = {b} Ω, maka tegangan V adalah {a*b:g} V.", f"{a*b:g} V"),
        "build_en": lambda a, b: (f"Ohm's law is expressed as V = I × R. If current I = {a} A and resistance R = {b} Ω, the voltage V is {a*b:g} V.", f"{a*b:g} V"),
        "q_formula_id": ["Apa rumus Hukum Ohm?", "Persamaan apa yang menyatakan Hukum Ohm?", "Bagaimana hubungan V, I, dan R dituliskan?", "Rumus apa yang digunakan pada Hukum Ohm?"],
        "q_formula_en": ["What is the formula for Ohm's law?", "Which equation expresses Ohm's law?", "How is the relationship between V, I, and R written?", "What formula is used for Ohm's law?"],
        "q_num_id": ["Berapa tegangan pada rangkaian tersebut?", "Berapa nilai tegangannya?", "Berapa hasil tegangan berdasarkan hukum Ohm?", "Berapakah tegangan V pada kasus tersebut?"],
        "q_num_en": ["What is the voltage in the circuit?", "What is the voltage value?", "What voltage is obtained using Ohm's law?", "What is the value of V in this case?"],
    },
    {
        "name_id": "energi kinetik", "name_en": "kinetic energy",
        "formula": "E_k = 1/2 × m × v²",
        "variants": [(4, 3), (2, 6), (5, 4), (8, 2)],
        "build": lambda a, b: (f"Energi kinetik dihitung dengan E_k = 1/2 × m × v². Jika massa m = {a} kg dan kecepatan v = {b} m/s, energi kinetiknya adalah {0.5*a*b*b:g} J.", f"{0.5*a*b*b:g} J"),
        "build_en": lambda a, b: (f"Kinetic energy is calculated using E_k = 1/2 × m × v². If mass m = {a} kg and speed v = {b} m/s, the kinetic energy is {0.5*a*b*b:g} J.", f"{0.5*a*b*b:g} J"),
        "q_formula_id": ["Apa rumus energi kinetik?", "Persamaan apa yang digunakan untuk menghitung energi kinetik?", "Bagaimana rumus energi kinetik dituliskan?", "Rumus apa yang digunakan untuk mencari energi kinetik?"],
        "q_formula_en": ["What is the kinetic energy formula?", "Which equation calculates kinetic energy?", "How is the kinetic energy formula written?", "What formula is used to find kinetic energy?"],
        "q_num_id": ["Berapa energi kinetik benda tersebut?", "Berapa nilai energi kinetiknya?", "Berapa hasil perhitungan energi kinetik?", "Berapakah energi kinetik pada kondisi tersebut?"],
        "q_num_en": ["What is the object's kinetic energy?", "What is the kinetic energy value?", "What kinetic energy is calculated?", "What is the kinetic energy in this case?"],
    },
    {
        "name_id": "energi potensial", "name_en": "potential energy",
        "formula": "E_p = m × g × h",
        "variants": [(2, 5), (3, 4), (4, 6), (5, 2)],
        "build": lambda a, b: (f"Energi potensial dirumuskan sebagai E_p = m × g × h. Jika massa m = {a} kg, percepatan gravitasi g = 10 m/s², dan tinggi h = {b} m, energi potensialnya adalah {a*10*b:g} J.", f"{a*10*b:g} J"),
        "build_en": lambda a, b: (f"Potential energy is given by E_p = m × g × h. If mass m = {a} kg, gravitational acceleration g = 10 m/s², and height h = {b} m, the potential energy is {a*10*b:g} J.", f"{a*10*b:g} J"),
        "q_formula_id": ["Apa rumus energi potensial?", "Persamaan apa yang digunakan untuk menghitung energi potensial?", "Bagaimana rumus energi potensial dituliskan?", "Rumus apa yang digunakan untuk mencari energi potensial?"],
        "q_formula_en": ["What is the potential energy formula?", "Which equation calculates potential energy?", "How is the potential energy formula written?", "What formula is used to find potential energy?"],
        "q_num_id": ["Berapa energi potensial benda tersebut?", "Berapa nilai energi potensialnya?", "Berapa hasil perhitungan energi potensial?", "Berapakah energi potensial pada kondisi tersebut?"],
        "q_num_en": ["What is the object's potential energy?", "What is the potential energy value?", "What potential energy is calculated?", "What is the potential energy in this case?"],
    },
    {
        "name_id": "usaha", "name_en": "work",
        "formula": "W = F × s",
        "variants": [(20, 3), (15, 4), (25, 2), (40, 5)],
        "build": lambda a, b: (f"Usaha dirumuskan dengan W = F × s. Jika gaya F = {a} N bekerja sejauh s = {b} m, usaha yang dilakukan adalah {a*b:g} J.", f"{a*b:g} J"),
        "build_en": lambda a, b: (f"Work is calculated by W = F × s. If force F = {a} N acts over s = {b} m, the work done is {a*b:g} J.", f"{a*b:g} J"),
        "q_formula_id": ["Apa rumus usaha?", "Persamaan apa yang digunakan untuk menghitung usaha?", "Bagaimana rumus usaha dituliskan?", "Rumus apa yang digunakan untuk mencari usaha?"],
        "q_formula_en": ["What is the work formula?", "Which equation calculates work?", "How is the work formula written?", "What formula is used to find work?"],
        "q_num_id": ["Berapa usaha yang dilakukan?", "Berapa nilai usahanya?", "Berapa hasil perhitungan usaha?", "Berapakah usaha pada kondisi tersebut?"],
        "q_num_en": ["What is the work done?", "What is the work value?", "What work is calculated?", "What is the work in this case?"],
    },
    {
        "name_id": "daya", "name_en": "power",
        "formula": "P = W/t",
        "variants": [(300, 5), (240, 4), (600, 10), (450, 9)],
        "build": lambda a, b: (f"Daya dirumuskan dengan P = W/t. Jika usaha W = {a} J dilakukan selama t = {b} s, daya yang dihasilkan adalah {a/b:g} W.", f"{a/b:g} W"),
        "build_en": lambda a, b: (f"Power is calculated by P = W/t. If work W = {a} J is done in t = {b} s, the power produced is {a/b:g} W.", f"{a/b:g} W"),
        "q_formula_id": ["Apa rumus daya?", "Persamaan apa yang digunakan untuk menghitung daya?", "Bagaimana rumus daya dituliskan?", "Rumus apa yang digunakan untuk mencari daya?"],
        "q_formula_en": ["What is the power formula?", "Which equation calculates power?", "How is the power formula written?", "What formula is used to find power?"],
        "q_num_id": ["Berapa daya yang dihasilkan?", "Berapa nilai dayanya?", "Berapa hasil perhitungan daya?", "Berapakah daya pada kondisi tersebut?"],
        "q_num_en": ["What is the power produced?", "What is the power value?", "What power is calculated?", "What is the power in this case?"],
    },
    {
        "name_id": "Hukum II Newton", "name_en": "Newton's second law",
        "formula": "F = m × a",
        "variants": [(4, 3), (5, 2), (6, 4), (8, 1.5)],
        "build": lambda a, b: (f"Hukum II Newton dinyatakan dengan F = m × a. Jika massa m = {a} kg dan percepatan a = {b} m/s², gaya yang dihasilkan adalah {a*b:g} N.", f"{a*b:g} N"),
        "build_en": lambda a, b: (f"Newton's second law is expressed as F = m × a. If mass m = {a} kg and acceleration a = {b} m/s², the resulting force is {a*b:g} N.", f"{a*b:g} N"),
        "q_formula_id": ["Apa rumus Hukum II Newton?", "Persamaan apa yang menyatakan Hukum II Newton?", "Bagaimana bentuk persamaan gaya pada Hukum II Newton?", "Rumus apa yang digunakan pada Hukum II Newton?"],
        "q_formula_en": ["What is the formula for Newton's second law?", "Which equation expresses Newton's second law?", "How is the force equation written for Newton's second law?", "What formula is used for Newton's second law?"],
        "q_num_id": ["Berapa gaya yang dihasilkan?", "Berapa nilai gaya pada kasus tersebut?", "Berapa hasil gaya berdasarkan Hukum II Newton?", "Berapakah gaya F pada kondisi tersebut?"],
        "q_num_en": ["What force is produced?", "What is the force value in this case?", "What force results from Newton's second law?", "What is the value of F in this case?"],
    },
]

# 19 tambahan: aljabar (3), kalkulus (5), statistika (3), kimia (4), ekonomi/keuangan (4).
# Digabung dengan 11 template yang sudah ada sehingga total menjadi 30.
FORMULA_SPECS.extend([
    _make_additional_formula_spec(
        "fungsi linear", "linear function", "y = ax + b",
        [(2, 3), (3, 4), (4, 2), (5, 1)], lambda a, b: a + b,
        "Fungsi linear dinyatakan dengan rumus {formula}. Untuk x = 1, koefisien a = {a} dan konstanta b = {b}, sehingga nilai y = a + b = {result}.",
        "A linear function is written as {formula}. For x = 1, coefficient a = {a} and intercept b = {b}, so y = a + b = {result}.",
    ),
    _make_additional_formula_spec(
        "identitas kuadrat jumlah", "square of a sum identity", "(a + b)^2 = a^2 + 2ab + b^2",
        [(2, 3), (3, 4), (4, 1), (5, 2)], lambda a, b: (a + b) ** 2,
        "Identitas aljabar ditulis {formula}. Untuk a = {a} dan b = {b}, nilai ruas kanan identitas tersebut adalah {result}.",
        "The algebraic identity is {formula}. For a = {a} and b = {b}, the right-hand side evaluates to {result}.",
    ),
    _make_additional_formula_spec(
        "selisih dua kuadrat", "difference of two squares", "a^2 - b^2 = (a - b)(a + b)",
        [(5, 3), (8, 2), (9, 4), (7, 1)], lambda a, b: a ** 2 - b ** 2,
        "Identitas selisih kuadrat adalah {formula}. Jika a = {a} dan b = {b}, nilai a^2 - b^2 adalah {result}.",
        "The difference-of-squares identity is {formula}. If a = {a} and b = {b}, the value of a^2 - b^2 is {result}.",
    ),
    _make_additional_formula_spec(
        "turunan fungsi linear", "derivative of a linear function", "d(ax + b)/dx = a",
        [(2, 3), (4, 7), (5, 1), (8, 2)], lambda a, b: a,
        "Turunan fungsi linear ditulis {formula}. Pada f(x) = {a}x + {b}, nilai turunannya konstan, yaitu {result}.",
        "The derivative of a linear function is {formula}. For f(x) = {a}x + {b}, the derivative is constant and equals {result}.",
    ),
    _make_additional_formula_spec(
        "turunan fungsi kuadrat", "derivative of a quadratic function", "d(ax^2 + bx)/dx = 2ax + b",
        [(1, 2), (2, 3), (3, 1), (4, 2)], lambda a, b: 2 * a + b,
        "Turunan fungsi kuadrat adalah {formula}. Untuk f(x) = {a}x^2 + {b}x pada x = 1, nilai turunannya adalah {result}.",
        "The derivative of a quadratic function is {formula}. For f(x) = {a}x^2 + {b}x at x = 1, the derivative value is {result}.",
    ),
    _make_additional_formula_spec(
        "turunan fungsi berpangkat", "power rule for derivatives", "d(x^n)/dx = n*x^(n-1)",
        [(2, 3), (3, 2), (4, 2), (5, 1)], lambda a, b: a * (b ** (a - 1)),
        r"Aturan pangkat untuk turunan adalah {formula}; dalam LaTeX ditulis $\frac{{d}}{{dx}}x^n = nx^{{n-1}}$. Untuk n = {a} dan x = {b}, hasil turunannya adalah {result}.",
        r"The power rule is {formula}; in LaTeX it is written as $\frac{{d}}{{dx}}x^n = nx^{{n-1}}$. For n = {a} and x = {b}, the derivative is {result}.",
    ),
    _make_additional_formula_spec(
        "integral fungsi berpangkat", "power rule for integration", "integral x^n dx = x^(n+1)/(n+1) + C",
        [(1, 2), (2, 3), (3, 2), (4, 1)], lambda a, b: (b ** (a + 1)) / (a + 1),
        r"Aturan integral pangkat ditulis {formula}; bentuk LaTeX-nya $\int x^n\,dx=\frac{{x^{{n+1}}}}{{n+1}}+C$. Untuk n = {a} dan x = {b}, nilai bagian x^(n+1)/(n+1) adalah {result}.",
        r"The power rule for integration is {formula}; its LaTeX form is $\int x^n\,dx=\frac{{x^{{n+1}}}}{{n+1}}+C$. For n = {a} and x = {b}, the x^(n+1)/(n+1) term is {result}.",
    ),
    _make_additional_formula_spec(
        "integral konstanta", "integral of a constant", "integral a dx = ax + C",
        [(2, 3), (3, 4), (4, 2), (5, 3)], lambda a, b: a * b,
        "Integral konstanta ditulis {formula}. Jika konstanta a = {a} dan batas evaluasi x = {b}, nilai suku ax adalah {result}.",
        "The integral of a constant is {formula}. For constant a = {a} and evaluation point x = {b}, the ax term equals {result}.",
    ),
    _make_additional_formula_spec(
        "rata-rata aritmetika", "arithmetic mean", "x_bar = sum(x_i)/n",
        [(30, 5), (40, 8), (60, 10), (72, 9)], lambda a, b: a / b,
        "Rata-rata aritmetika dihitung dengan {formula}. Jika jumlah seluruh data adalah {a} dan banyak data n = {b}, nilai rata-ratanya {result}.",
        "The arithmetic mean uses {formula}. If the total of the observations is {a} and the number of observations n = {b}, the mean is {result}.",
    ),
    _make_additional_formula_spec(
        "varians populasi", "population variance", "sigma^2 = sum((x_i - mu)^2)/N",
        [(20, 5), (36, 6), (45, 9), (64, 8)], lambda a, b: a / b,
        "Varians populasi ditulis {formula}. Jika jumlah kuadrat selisih terhadap mean adalah {a} dan jumlah data N = {b}, variansnya {result}.",
        "Population variance is {formula}. If the sum of squared deviations from the mean is {a} and the population size N = {b}, the variance is {result}.",
    ),
    _make_additional_formula_spec(
        "simpangan baku populasi", "population standard deviation", "sigma = sqrt(sigma^2)",
        [(4, 2), (9, 3), (16, 4), (25, 5)], lambda a, b: math.sqrt(a),
        "Simpangan baku populasi dihitung dengan {formula}. Jika varians sigma^2 = {a}, simpangan bakunya adalah {result}; parameter b = {b} disertakan sebagai nilai pembanding.",
        "Population standard deviation is computed using {formula}. If variance sigma^2 = {a}, the standard deviation is {result}; b = {b} is included as a comparison value.",
    ),
    _make_additional_formula_spec(
        "persamaan gas ideal", "ideal gas equation", "PV = nRT",
        [(2, 3), (4, 5), (6, 2), (8, 4)], lambda a, b: a * b,
        "Persamaan gas ideal adalah {formula}. Untuk tekanan P = {a} atm dan volume V = {b} L, hasil kali tekanan-volume PV adalah {result} atm L.",
        "The ideal gas equation is {formula}. For pressure P = {a} atm and volume V = {b} L, the pressure-volume product PV is {result} atm L.",
        unit="atm L",
    ),
    _make_additional_formula_spec(
        "jumlah mol zat", "amount of substance in moles", "n = m/M",
        [(10, 2), (18, 6), (20, 4), (30, 5)], lambda a, b: a / b,
        "Jumlah mol dihitung dengan {formula}. Jika massa zat m = {a} g dan massa molar M = {b} g/mol, jumlah zatnya {result} mol.",
        "Amount in moles is calculated using {formula}. If mass m = {a} g and molar mass M = {b} g/mol, the amount of substance is {result} mol.",
        unit="mol",
    ),
    _make_additional_formula_spec(
        "molaritas larutan", "solution molarity", "C = n/V",
        [(2, 4), (3, 6), (5, 10), (6, 3)], lambda a, b: a / b,
        "Molaritas dihitung dengan {formula}. Jika jumlah mol zat n = {a} mol dan volume larutan V = {b} L, konsentrasi molarnya {result} mol/L.",
        "Molarity is calculated using {formula}. If amount n = {a} mol and solution volume V = {b} L, the molar concentration is {result} mol/L.",
        unit="mol/L",
    ),
    _make_additional_formula_spec(
        "skala pH", "pH scale", "pH = -log10[H+]",
        [(1, 3), (2, 4), (5, 6), (1, 5)], lambda a, b: b - math.log10(a),
        "Skala pH menggunakan {formula}. Jika konsentrasi ion hidrogen [H+] = {a} × 10^-{b} M, nilai pH adalah {result}.",
        "The pH scale uses {formula}. If hydrogen ion concentration [H+] = {a} × 10^-{b} M, the pH is {result}.",
    ),
    _make_additional_formula_spec(
        "bunga sederhana", "simple interest", "I = P*r*t",
        [(100000, 5), (250000, 4), (500000, 3), (200000, 8)], lambda a, b: a * b / 100,
        "Bunga sederhana dihitung dengan {formula}. Jika pokok pinjaman P = {a} rupiah, suku bunga r = {b}% per tahun, dan t = 1 tahun, bunganya {result} rupiah.",
        "Simple interest is calculated using {formula}. If principal P = {a} rupiah, annual rate r = {b}%, and t = 1 year, the interest is {result} rupiah.",
        unit="rupiah",
    ),
    _make_additional_formula_spec(
        "bunga majemuk dua tahun", "two-year compound interest", "A = P(1 + r)^t",
        [(100000, 5), (200000, 4), (300000, 10), (500000, 2)], lambda a, b: a * (1 + b / 100) ** 2,
        "Nilai akhir dengan bunga majemuk ditulis {formula}. Jika modal awal P = {a} rupiah, bunga tahunan r = {b}%, dan t = 2 tahun, nilai akhirnya {result} rupiah.",
        "The final amount with compound interest is {formula}. If initial principal P = {a} rupiah, annual rate r = {b}%, and t = 2 years, the final amount is {result} rupiah.",
        unit="rupiah",
    ),
    _make_additional_formula_spec(
        "laba usaha", "business profit", "profit = revenue - cost",
        [(100, 60), (250, 150), (500, 320), (800, 550)], lambda a, b: a - b,
        "Laba usaha dihitung dengan {formula}. Jika pendapatan adalah {a} ribu rupiah dan biaya adalah {b} ribu rupiah, laba yang diperoleh {result} ribu rupiah.",
        "Business profit is calculated using {formula}. If revenue is {a} thousand rupiah and cost is {b} thousand rupiah, profit is {result} thousand rupiah.",
        unit="ribu rupiah", unit_en="thousand rupiah",
    ),
    _make_additional_formula_spec(
        "jumlah produksi titik impas", "break-even quantity", "Q = F/(P - V)",
        [(100, 2), (200, 5), (150, 4), (300, 6)], lambda a, b: a / (10 - b),
        "Jumlah produksi titik impas menggunakan {formula}. Jika biaya tetap F = {a}, harga per unit P = 10, dan biaya variabel per unit V = {b}, kuantitas titik impasnya {result} unit.",
        "Break-even quantity uses {formula}. If fixed cost F = {a}, unit price P = 10, and variable cost per unit V = {b}, the break-even quantity is {result} units.",
        unit="unit",
    ),
])

if len(FORMULA_SPECS) < 30:
    raise RuntimeError(f"Jumlah template formula kurang dari 30: {len(FORMULA_SPECS)}")
print(f"Template formula aktif: {len(FORMULA_SPECS)} (minimum 30).", flush=True)


def generate_formula_data(train_target, eval_target, seed):
    rng = random.Random(seed)

    def build_from_variant_slice(var_slice, target_count, source_tag):
        raw_rows = []
        for spec in FORMULA_SPECS:
            for a, b in spec["variants"][var_slice]:
                ctx_id, num_id = spec["build"](a, b)
                ctx_en, num_en = spec["build_en"](a, b)
                for q_idx in range(4):
                    raw_rows.extend([
                        {
                            "lang": "id", "context": ctx_id, "answer": spec["formula"],
                            "question": spec["q_formula_id"][q_idx], "answer_start": ctx_id.find(spec["formula"]),
                            "answer_type": "formula", "has_formula": True,
                            "formula_spans": json.dumps([spec["formula"]], ensure_ascii=False), "source": source_tag
                        },
                        {
                            "lang": "en", "context": ctx_en, "answer": spec["formula"],
                            "question": spec["q_formula_en"][q_idx], "answer_start": ctx_en.find(spec["formula"]),
                            "answer_type": "formula", "has_formula": True,
                            "formula_spans": json.dumps([spec["formula"]], ensure_ascii=False), "source": source_tag
                        },
                        {
                            "lang": "id", "context": ctx_id, "answer": num_id,
                            "question": spec["q_num_id"][q_idx], "answer_start": ctx_id.find(num_id),
                            "answer_type": "number", "has_formula": True,
                            "formula_spans": json.dumps([spec["formula"]], ensure_ascii=False), "source": source_tag
                        },
                        {
                            "lang": "en", "context": ctx_en, "answer": num_en,
                            "question": spec["q_num_en"][q_idx], "answer_start": ctx_en.find(num_en),
                            "answer_type": "number", "has_formula": True,
                            "formula_spans": json.dumps([spec["formula"]], ensure_ascii=False), "source": source_tag
                        }
                    ])
        rng.shuffle(raw_rows)
        id_part = [r for r in raw_rows if r["lang"] == "id"][:target_count]
        en_part = [r for r in raw_rows if r["lang"] == "en"][:target_count]
        return id_part, en_part

    tr_id, tr_en = build_from_variant_slice(slice(0, 3), train_target, "synthetic_formula_train")
    ev_id, ev_en = build_from_variant_slice(slice(3, 4), eval_target, "synthetic_formula_eval")
    return tr_id, tr_en, ev_id, ev_en


# ----------------------------- 5. Formula data generation -----------------------------
print("\n" + "=" * 78, flush=True)
print("[5/8] MEMBANGUN SYNTHETIC FORMULA DATASET", flush=True)
print("=" * 78, flush=True)

formula_train = {"id": [], "en": []}
formula_eval = {"id": [], "en": []}

if USE_FORMULA_AUGMENTATION:
    f_start = time.time()
    f_tr_id, f_tr_en, f_ev_id, f_ev_en = generate_formula_data(
        FORMULA_TRAIN_PER_LANG,
        FORMULA_EVAL_PER_LANG,
        SEED
    )
    formula_train["id"], formula_train["en"] = f_tr_id, f_tr_en
    formula_eval["id"], formula_eval["en"] = f_ev_id, f_ev_en

    train_f_ctx = {r["context"] for r in formula_train["id"] + formula_train["en"]}
    eval_f_ctx = {r["context"] for r in formula_eval["id"] + formula_eval["en"]}
    assert train_f_ctx.isdisjoint(eval_f_ctx), "Konteks formula bocor antara train dan eval!"

    print(f"Formula generation selesai dalam {time.time() - f_start:.1f} detik.", flush=True)
    print(f"Formula Train: ID={len(formula_train['id'])}, EN={len(formula_train['en'])}", flush=True)
    print(f"Formula Eval : ID={len(formula_eval['id'])}, EN={len(formula_eval['en'])}", flush=True)

# ----------------------------- 6. Gabungkan Data -----------------------------
print("\n" + "=" * 78, flush=True)
print("[6/8] MENGGABUNGKAN DATA GENERAL + FORMULA KE TRAIN", flush=True)
print("=" * 78, flush=True)

train_rows = [row.copy() for row in train_general]
if USE_FORMULA_AUGMENTATION:
    train_rows.extend(row.copy() for row in formula_train["id"])
    train_rows.extend(row.copy() for row in formula_train["en"])

# Transform exactly 20% of training examples to a slide-like presentation style.
# Validation/test remain untouched so scores stay comparable to the source datasets.
slide_target = int(round(len(train_rows) * 0.20))
slide_indices = rng.sample(range(len(train_rows)), slide_target) if slide_target else []
slide_transformed = 0
for row_idx in slide_indices:
    row = train_rows[row_idx]
    styled_context = convert_to_slide_style(row["context"], row["answer"])
    new_start = styled_context.find(row["answer"])
    if new_start < 0:
        continue
    row["context"] = styled_context
    row["answer_start"] = new_start
    row["has_formula"] = detect_formula_content(styled_context)[0]
    row["formula_spans"] = json.dumps([x[2] for x in detect_formula_spans(styled_context)], ensure_ascii=False)
    row["source"] = str(row.get("source", "unknown")) + "_slide_style"
    slide_transformed += 1

print(f"Slide-style augmentation: {slide_transformed:,}/{len(train_rows):,} rows "
      f"({slide_transformed / max(1, len(train_rows)):.1%}); target 20%.", flush=True)
rng.shuffle(train_rows)
rng.shuffle(val_general)

print(f"Train total      : {len(train_rows):,}", flush=True)
print(f"Validation total : {len(val_general):,}", flush=True)
print(f"Formula eval ID  : {len(formula_eval['id']):,}", flush=True)
print(f"Formula eval EN  : {len(formula_eval['en']):,}", flush=True)

# ----------------------------- 7. Unit Test & Leakage Check -----------------------------
print("\n" + "=" * 78, flush=True)
print("[7/8] UNIT TEST FORMULA DETECTOR + CEK LEAKAGE", flush=True)
print("=" * 78, flush=True)

train_contexts = {r["context"] for r in train_general}
val_contexts = {r["context"] for r in val_general}
assert train_contexts.isdisjoint(val_contexts), "Context leakage terjadi antara train dan validation umum."
print(f"Context leakage umum: 0 overlap (train={len(train_contexts):,}, val={len(val_contexts):,})", flush=True)

_formula_tests = [
    "Rumus luas adalah A = p × l.",
    "Energi kinetik E_k = 1/2 × m × v².",
    "Kecepatan v = s/t dan massa jenis ρ = m/V.",
    "Pembelajaran berlangsung di ruang kelas pada pagi hari.",
    "Luas persegi panjang tersebut adalah 40 cm².",
]
print("\nUnit test formula detector & answer type:")
for t in _formula_tests:
    flg, spns = detect_formula_content(t)
    print(f"  flag={str(flg):<5} | type={answer_type(t):<7} | text={t}")

# ----------------------------- 8. Finalisasi Dataset HF -----------------------------
print("\n" + "=" * 78, flush=True)
print("[8/8] FINALISASI DATASET HUGGING FACE", flush=True)
print("=" * 78, flush=True)

raw_train = Dataset.from_list(train_rows)
raw_val = Dataset.from_list(val_general)
raw_formula_eval = (
    Dataset.from_list(formula_eval["id"] + formula_eval["en"])
    if USE_FORMULA_AUGMENTATION
    else None
)

print(f"raw_train        : {len(raw_train):,} rows", flush=True)
print(f"raw_val          : {len(raw_val):,} rows", flush=True)
print(f"raw_formula_eval : {len(raw_formula_eval):,} rows", flush=True)
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized(), "(harus False)", flush=True)
print("Cell 3 selesai. Lanjutkan ke Cell 4.", flush=True)

## Cell 4 — Tokenizer, answer markers, context window, dan tokenisasi

Perbaikan inti terhadap blind spot Q/A ada di sini.

Prompt baru secara konseptual menjadi:

`generate question {lang}: answer_type: {type}: formula_context: {yes/no}: answer: {answer} context: ... <answer>answer</answer> ...`

Ada dua sinyal answer-control sekaligus:

- answer diberikan eksplisit sebelum context;
- kemunculan answer yang benar di dalam context diberi marker `<answer> ... </answer>`.

Jika context melebihi 512 token, jendela dipersempit di sekitar answer, lalu ditokenisasi ulang sampai muat. Ini jauh lebih aman untuk dokumen panjang daripada sekadar memotong 512 token pertama.

In [ ]:
SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<answer>",
        "</answer>",
        "<formula>",
        "</formula>",
    ]
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
added = tokenizer.add_special_tokens(SPECIAL_TOKENS)

print("Tokenizer:", type(tokenizer).__name__)
print("Vocab setelah special token:", len(tokenizer))
print("Token tambahan:", added)
print("pad/eos:", tokenizer.pad_token_id, "/", tokenizer.eos_token_id)


def _span_overlaps(start, end, answer_span):
    if answer_span is None:
        return False
    a_start, a_end = answer_span
    return start < a_end and end > a_start


def mark_formulas(context, answer_span=None):
    """Bungkus formula dengan token khusus, kecuali span yang overlap dengan answer."""
    text = str(context)
    spans = [span for span in detect_formula_spans(text)
             if not _span_overlaps(span[0], span[1], answer_span)]
    for start, end, _ in sorted(spans, key=lambda item: item[0], reverse=True):
        text = text[:start] + "<formula>" + text[start:end] + "</formula>" + text[end:]
    return text


def mark_answer(context, answer, answer_start):
    """Wrap answer without dropping any suffix after the marked span."""
    context = str(context)
    answer = str(answer)
    try:
        start = int(answer_start)
    except (TypeError, ValueError):
        start = -1
    if not answer:
        return context
    if start < 0 or context[start:start + len(answer)].casefold() != answer.casefold():
        start = context.casefold().find(answer.casefold())
    if start < 0:
        return context
    end = start + len(answer)
    return context[:start] + "<answer>" + context[start:end] + "</answer>" + context[end:]


def mark_answer_and_formulas(context, answer, answer_start):
    """Apply formula markers first, then adjust answer offset for wrappers before it."""
    text = str(context)
    answer = str(answer)
    try:
        start = int(answer_start)
    except (TypeError, ValueError):
        start = -1
    if start < 0 or text[start:start + len(answer)].casefold() != answer.casefold():
        start = text.casefold().find(answer.casefold()) if answer else -1
    if start < 0:
        return mark_formulas(text, None)
    end = start + len(answer)
    formula_spans = [span for span in detect_formula_spans(text)
                     if not _span_overlaps(span[0], span[1], (start, end))]
    marked_context = mark_formulas(text, (start, end))
    shift = sum(len("<formula>") + len("</formula>")
                for span_start, span_end, _ in formula_spans if span_end <= start)
    return mark_answer(marked_context, answer, start + shift)


def make_prompt(lang, answer, answer_kind, has_formula, marked_context):
    return (
        f"generate question {lang}: "
        f"answer_type: {answer_kind}: "
        f"formula_context: {'yes' if has_formula else 'no'}: "
        f"answer: {answer} "
        f"context: {marked_context}"
    )


def make_length_safe_prompt(lang, context, answer, answer_kind, has_formula, answer_start):
    marked_full = mark_answer_and_formulas(context, answer, answer_start)
    prompt_full = make_prompt(lang, answer, answer_kind, has_formula, marked_full)
    full_ids = tokenizer(prompt_full, truncation=False)["input_ids"]
    if len(full_ids) <= MAX_INPUT_LENGTH:
        return prompt_full

    # Jendela karakter dimulai cukup lebar, lalu dipersempit di sekitar answer.
    start = max(0, int(answer_start) if answer_start >= 0 else len(context) // 2)
    left = max(0, start - 2500)
    right = min(len(context), start + len(answer) + 2500)

    for _ in range(10):
        window_context = context[left:right]
        local_start = context.find(answer, left, right)
        if local_start < 0:
            local_start = max(left, min(start, right - len(answer)))
        local_answer_start = local_start - left
        marked_window = mark_answer_and_formulas(window_context, answer, local_answer_start)
        prompt = make_prompt(lang, answer, answer_kind, has_formula, marked_window)
        ids = tokenizer(prompt, truncation=False)["input_ids"]
        if len(ids) <= MAX_INPUT_LENGTH:
            return prompt

        center = (left + right) // 2
        half = max(250, int((right - left) * 0.40))
        left = max(0, center - half)
        right = min(len(context), center + half)

    # Fallback terakhir: tokenizer truncation, tetapi tetap meletakkan answer terlebih dahulu.
    return prompt_full


def tokenize_dataset(dataset_obj):
    def process(batch):
        prompts = []
        for lang, context, answer, kind, has_formula, start in zip(
            batch["lang"],
            batch["context"],
            batch["answer"],
            batch["answer_type"],
            batch["has_formula"],
            batch["answer_start"],
        ):
            prompts.append(
                make_length_safe_prompt(
                    lang,
                    context,
                    answer,
                    kind,
                    bool(has_formula),
                    int(start),
                )
            )

        enc = tokenizer(
            prompts,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
        )
        labels = tokenizer(
            text_target=batch["question"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        )
        enc["labels"] = labels["input_ids"]
        return enc

    return dataset_obj.map(
        process,
        batched=True,
        batch_size=128,
        remove_columns=dataset_obj.column_names,
        desc="Tokenisasi answer-aware",
    )


# Filter ekstrem setelah tokenizer: answer/target terlalu panjang dapat membuat
# conditioning tidak lengkap. Dataset umumnya kehilangan sangat sedikit baris di sini.
def filter_length_extremes(dataset_obj):
    answer_lengths = []
    question_lengths = []
    keep = []
    for idx, row in enumerate(dataset_obj):
        a_len = len(tokenizer(row["answer"], add_special_tokens=False)["input_ids"])
        q_len = len(tokenizer(row["question"], add_special_tokens=False)["input_ids"])
        answer_lengths.append(a_len)
        question_lengths.append(q_len)
        if a_len <= MAX_ANSWER_TOKENS and q_len <= MAX_TARGET_LENGTH:
            keep.append(idx)
    filtered = dataset_obj.select(keep)
    print(
        f"Filter panjang: sebelum={len(dataset_obj)}, sesudah={len(filtered)}, "
        f"answer > {MAX_ANSWER_TOKENS} atau question > {MAX_TARGET_LENGTH} dibuang={len(dataset_obj)-len(filtered)}"
    )
    print(
        f"Panjang answer: mean={np.mean(answer_lengths):.2f}, p95={np.percentile(answer_lengths,95):.1f}, max={max(answer_lengths)}"
    )
    print(
        f"Panjang target: mean={np.mean(question_lengths):.2f}, p95={np.percentile(question_lengths,95):.1f}, max={max(question_lengths)}"
    )
    return filtered


raw_train = filter_length_extremes(raw_train)
raw_val = filter_length_extremes(raw_val)
if raw_formula_eval is not None:
    raw_formula_eval = filter_length_extremes(raw_formula_eval)

tok_train = tokenize_dataset(raw_train)
tok_val = tokenize_dataset(raw_val)
tok_formula_eval = tokenize_dataset(raw_formula_eval) if raw_formula_eval is not None else None

for name, ds in (("train", tok_train), ("validation", tok_val), ("formula_eval", tok_formula_eval)):
    if ds is None:
        continue
    lengths = np.array([len(row) for row in ds["input_ids"]])
    print(f"[{name}] n={len(ds)}, input mean={lengths.mean():.1f}, p95={np.percentile(lengths,95):.0f}, max={lengths.max()}")
    assert lengths.max() <= MAX_INPUT_LENGTH

print("\nDecoded sample:")
print(tokenizer.decode(tok_train[0]["input_ids"], skip_special_tokens=False)[:900])
print("Target:", tokenizer.decode(tok_train[0]["labels"], skip_special_tokens=False))
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized(), "(harus False)")


## Cell 5 — Data collator, metric ROUGE/BLEU, model configuration, dan Seq2SeqTrainer

mT5-base lebih besar daripada mT5-small, jadi default batch diturunkan menjadi 2 per GPU dan gradient accumulation dinaikkan sehingga effective batch tetap 16.

Parameter inti tetap mengikuti eksperimen awal:

- LR `3e-4`
- warmup `0.1`
- AdamW
- weight decay `0.01`
- label smoothing `0.1`
- 3 epoch
- beam search 4
- evaluation/checkpoint setiap 250 optimizer steps
- best checkpoint berdasarkan `rougeL`
- FP16 `False`
- gradient checkpointing `True`

In [ ]:
from accelerate import notebook_launcher

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")


def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    pad_id = tokenizer.pad_token_id
    vocab_size = len(tokenizer)

    predictions = np.where(
        (predictions < 0) | (predictions >= vocab_size),
        pad_id,
        predictions,
    )
    labels = np.where(labels == -100, pad_id, labels)
    labels = np.where(
        (labels < 0) | (labels >= vocab_size),
        pad_id,
        labels,
    )

    decoded_preds = [normalize_space(x) for x in tokenizer.batch_decode(predictions, skip_special_tokens=True)]
    decoded_labels = [normalize_space(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    decoded_preds = [x if x else "<empty>" for x in decoded_preds]
    decoded_labels = [x if x else "<empty>" for x in decoded_labels]

    rouge = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        rouge_types=["rouge1", "rouge2", "rougeL"],
        use_stemmer=False,
    )
    bleu = bleu_metric.compute(
        predictions=decoded_preds,
        references=[[x] for x in decoded_labels],
        max_order=4,
        smooth=True,
    )

    lengths = np.count_nonzero(predictions != pad_id, axis=1)
    return {
        "rouge1": round(float(rouge["rouge1"]) * 100, 4),
        "rouge2": round(float(rouge["rouge2"]) * 100, 4),
        "rougeL": round(float(rouge["rougeL"]) * 100, 4),
        "bleu4": round(float(bleu["bleu"]) * 100, 4),
        "gen_len": round(float(lengths.mean()), 2),
    }

import dataclasses
import math

def make_training_objects():
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.resize_token_embeddings(len(tokenizer))
    model.config.use_cache = False

    model.generation_config.num_beams = 4
    model.generation_config.length_penalty = 1.0
    model.generation_config.no_repeat_ngram_size = 2
    model.generation_config.max_length = MAX_TARGET_LENGTH

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8,
    )

    # Hitung total langkah dan warmup_steps secara eksplisit
    total_steps = int(math.ceil(len(tok_train) / TARGET_EFFECTIVE_BATCH_SIZE) * NUM_EPOCHS)
    calculated_warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    # Daftar konfigurasi lengkap
    raw_args = {
        "output_dir": OUTPUT_DIR,
        "num_train_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler_type": "linear",
        "warmup_steps": calculated_warmup_steps,  # Menggantikan warmup_ratio
        "weight_decay": WEIGHT_DECAY,
        "optim": "adamw_torch",
        "label_smoothing_factor": LABEL_SMOOTHING,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "eval_strategy": "steps",
        "eval_steps": EVAL_STEPS,
        "save_strategy": "steps",
        "save_steps": EVAL_STEPS,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "rougeL",
        "greater_is_better": True,
        "predict_with_generate": True,
        "generation_num_beams": 4,
        "generation_max_length": MAX_TARGET_LENGTH,
        "fp16": False,
        "bf16": False,
        "gradient_checkpointing": True,
        "max_grad_norm": 1.0,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "logging_nan_inf_filter": True,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "dataloader_num_workers": 2,
        "dataloader_pin_memory": True,
        "remove_unused_columns": True,
        "ddp_backend": "nccl",
        "ddp_find_unused_parameters": False,
        "ddp_timeout": 7200,
    }

    # Filter otomatis: abaikan keyword yang tidak didukung versi Transformers saat ini
    if dataclasses.is_dataclass(Seq2SeqTrainingArguments):
        valid_fields = {f.name for f in dataclasses.fields(Seq2SeqTrainingArguments)}
        safe_args = {k: v for k, v in raw_args.items() if k in valid_fields}
    else:
        safe_args = raw_args

    training_args = Seq2SeqTrainingArguments(**safe_args)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    return trainer

def train_worker():
    # Semua seed PyTorch/CUDA baru dinyalakan di worker setelah DDP dibuat.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    trainer = make_training_objects()
    is_main = trainer.is_world_process_zero()

    if is_main:
        print("\n===== TRAIN CONFIG =====")
        print("Model              :", MODEL_NAME)
        print("GPU worker         :", torch.cuda.get_device_name(torch.cuda.current_device()))
        print("World size         :", trainer.args.world_size)
        print("Batch/GPU          :", PER_DEVICE_TRAIN_BATCH_SIZE)
        print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
        print("Effective batch    :", PER_DEVICE_TRAIN_BATCH_SIZE * NUM_GPUS * GRADIENT_ACCUMULATION_STEPS)
        print("Learning rate      :", LEARNING_RATE)
        print("Warmup ratio       :", WARMUP_RATIO)
        print("Label smoothing    :", LABEL_SMOOTHING)
        print("FP16               : False")
        print("Gradient checkpoint:", True)
        print("Training rows      :", len(tok_train))
        print("Validation rows    :", len(tok_val))

    train_result = trainer.train()

    # Setelah train selesai, best checkpoint sudah dimuat karena load_best_model_at_end=True.
    trainer.save_model(FINAL_DIR)
    if is_main:
        tokenizer.save_pretrained(FINAL_DIR)

    # Evaluasi utama dan formula benchmark dilakukan dengan model terbaik.
    metrics_general = trainer.evaluate(
        eval_dataset=tok_val,
        metric_key_prefix="eval",
    )

    metrics_formula = None
    if tok_formula_eval is not None:
        metrics_formula = trainer.evaluate(
            eval_dataset=tok_formula_eval,
            metric_key_prefix="formula_eval",
        )

    if is_main:
        os.makedirs(FINAL_DIR, exist_ok=True)
        result = {
            "training": train_result.metrics,
            "general_validation": metrics_general,
            "formula_validation": metrics_formula,
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "best_metric": trainer.state.best_metric,
            "config": {
                "model": MODEL_NAME,
                "num_gpus": NUM_GPUS,
                "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
                "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
                "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * NUM_GPUS * GRADIENT_ACCUMULATION_STEPS,
                "learning_rate": LEARNING_RATE,
                "warmup_ratio": WARMUP_RATIO,
                "weight_decay": WEIGHT_DECAY,
                "label_smoothing_factor": LABEL_SMOOTHING,
                "epochs": NUM_EPOCHS,
                "fp16": False,
                "gradient_checkpointing": True,
                "max_input_length": MAX_INPUT_LENGTH,
                "max_target_length": MAX_TARGET_LENGTH,
                "num_beams": 4,
            },
        }
        with open(os.path.join(FINAL_DIR, "training_metrics.json"), "w", encoding="utf-8") as handle:
            json.dump(result, handle, indent=2, ensure_ascii=False, default=float)
        pd.DataFrame(trainer.state.log_history).to_csv(
            os.path.join(FINAL_DIR, "log_history.csv"),
            index=False,
        )
        print("\n[worker] training selesai:", FINAL_DIR)


print("Trainer worker siap.")
print("[cek] CUDA aktif proses utama:", torch.cuda.is_initialized(), "(harus False)")


## Cell 6 — Training DDP T4 × 2, evaluasi, dan answer-sensitivity audit

Cell ini menjalankan training dengan **dua proses DDP**, satu proses per T4.

Setelah training selesai:

1. model terbaik dimuat melalui `load_best_model_at_end=True`;
2. validation umum dievaluasi dengan ROUGE/BLEU;
3. formula test dievaluasi terpisah;
4. model terbaik dimuat pada GPU 0 untuk **answer-sensitivity audit**.

Audit terakhir sangat penting: untuk context yang sama, dua answer berbeda harus menghasilkan pertanyaan yang berubah. Jika dua output identik terus-menerus, berarti model masih mengabaikan answer.

In [ ]:
from accelerate import notebook_launcher

if torch.cuda.is_initialized():
    raise RuntimeError(
        "CUDA sudah aktif di proses utama sebelum DDP. Restart Session Kaggle dan jalankan notebook dari Cell 1."
    )

LAUNCH_PORT = "29525"
print(f"Meluncurkan DDP dengan {NUM_GPUS} GPU pada port {LAUNCH_PORT} ...")
notebook_launcher(
    train_worker,
    args=(),
    num_processes=NUM_GPUS,
    mixed_precision="no",
    use_port=LAUNCH_PORT,
)

metrics_path = os.path.join(FINAL_DIR, "training_metrics.json")
if not os.path.exists(metrics_path):
    raise FileNotFoundError("training_metrics.json tidak ditemukan. Periksa traceback training.")

with open(metrics_path, "r", encoding="utf-8") as handle:
    training_metrics = json.load(handle)

print("\n===== HASIL TRAINING =====")
print("Best checkpoint:", training_metrics["best_checkpoint"])
print("Best rougeL    :", training_metrics["best_metric"])
print("Training metrics:")
for key, value in training_metrics["training"].items():
    print(f"  {key}: {value}")

print("\nGeneral validation:")
for key, value in training_metrics["general_validation"].items():
    print(f"  {key}: {value}")

print("\nFormula validation:")
if training_metrics["formula_validation"] is None:
    print("  Formula augmentation disabled.")
else:
    for key, value in training_metrics["formula_validation"].items():
        print(f"  {key}: {value}")

# --------------------------- Answer sensitivity audit ---------------------------
# Audit hanya dijalankan setelah DDP selesai; sekarang proses utama boleh memakai cuda:0.
from transformers import AutoModelForSeq2SeqLM

inference_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
audit_tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
audit_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(inference_device)
audit_model.eval()


def audit_generate(context, answer, lang, kind, has_formula, answer_start):
    marked = mark_answer(context, answer, answer_start)
    prompt = make_length_safe_prompt(lang, context, answer, kind, has_formula, answer_start)
    enc = audit_tokenizer(
        prompt,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(inference_device)
    with torch.inference_mode():
        ids = audit_model.generate(
            **enc,
            num_beams=4,
            num_return_sequences=1,
            length_penalty=1.0,
            no_repeat_ngram_size=2,
            max_length=MAX_TARGET_LENGTH,
        )
    return normalize_space(audit_tokenizer.decode(ids[0], skip_special_tokens=True))


# Ambil konteks validation yang memiliki minimal dua QA dengan answer berbeda.
groups = defaultdict(list)
for row in raw_val:
    groups[row["context"]].append(row)

pairs = []
for context, rows in groups.items():
    distinct = {}
    for row in rows:
        distinct[row["answer"]] = row
    if len(distinct) >= 2:
        vals = list(distinct.values())
        pairs.append((vals[0], vals[1]))
    if len(pairs) >= 30:
        break

sensitivity_records = []
for row_a, row_b in pairs:
    q_a = audit_generate(
        row_a["context"], row_a["answer"], row_a["lang"], row_a["answer_type"], bool(row_a["has_formula"]), row_a["answer_start"]
    )
    q_b = audit_generate(
        row_b["context"], row_b["answer"], row_b["lang"], row_b["answer_type"], bool(row_b["has_formula"]), row_b["answer_start"]
    )
    same = q_a.casefold() == q_b.casefold()
    sensitivity_records.append({
        "lang": row_a["lang"],
        "answer_a": row_a["answer"],
        "question_a": q_a,
        "answer_b": row_b["answer"],
        "question_b": q_b,
        "identical_output": same,
    })

if sensitivity_records:
    changed = sum(not row["identical_output"] for row in sensitivity_records)
    sensitivity_rate = changed / len(sensitivity_records)
else:
    sensitivity_rate = None

print("\n===== ANSWER SENSITIVITY AUDIT =====")
print("Pairs tested           :", len(sensitivity_records))
print("Changed output rate    :", sensitivity_rate)
print("Identical output count :", sum(r["identical_output"] for r in sensitivity_records))

for record in sensitivity_records[:8]:
    print("\nLANG:", record["lang"])
    print("A:", record["answer_a"])
    print("Q(A):", record["question_a"])
    print("B:", record["answer_b"])
    print("Q(B):", record["question_b"])
    print("IDENTICAL:", record["identical_output"])

with open(os.path.join(FINAL_DIR, "answer_sensitivity_audit.json"), "w", encoding="utf-8") as handle:
    json.dump(
        {
            "pairs_tested": len(sensitivity_records),
            "changed_output_rate": sensitivity_rate,
            "records": sensitivity_records,
        },
        handle,
        indent=2,
        ensure_ascii=False,
    )

# Simpan metric tambahan tanpa mencampurnya dengan ROUGE/BLEU best-checkpoint metric.
training_metrics["answer_sensitivity"] = {
    "pairs_tested": len(sensitivity_records),
    "changed_output_rate": sensitivity_rate,
    "identical_output_count": sum(r["identical_output"] for r in sensitivity_records),
}
with open(metrics_path, "w", encoding="utf-8") as handle:
    json.dump(training_metrics, handle, indent=2, ensure_ascii=False, default=float)

print("\nSemua hasil tersimpan di:", FINAL_DIR)


## Cell 7 — Inferensi, formula detector, answer-type alignment, dan uji manual

Cell ini memuat model terbaik dan menyediakan:

- `detect_formula_content(text)` untuk mendeteksi material berisi rumus sebelum QG;
- `generate_question(context, answer, lang)` untuk QG biasa;
- `show_qg(...)` untuk memperlihatkan context, answer, tipe answer, formula flag, dan hasil pertanyaan;
- `answer_type_alignment(question, answer, lang)` sebagai diagnostic sederhana, bukan pengganti evaluasi manusia;
- `interactive_qg()` untuk pengujian manual.

Untuk dokumen PDF/PPT di fase berikutnya, alur yang disarankan adalah: **extract → formula detection → preserve formula spans → candidate answer extraction → QG**. Jangan memaksa mT5 menjadi detector dokumen sekaligus generator pertanyaan.

In [ ]:
import re
import os
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

INFER_DIR = FINAL_DIR
INFER_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

infer_tokenizer = AutoTokenizer.from_pretrained(INFER_DIR)
infer_model = AutoModelForSeq2SeqLM.from_pretrained(INFER_DIR).to(INFER_DEVICE)
infer_model.eval()


def answer_type_alignment(question, answer, lang):
    """
    Diagnostic sederhana untuk melihat apakah bentuk pertanyaan konsisten dengan tipe answer.
    Ini bukan metric ilmiah universal; gunakan sebagai error-analysis signal.
    """
    q = normalize_space(question).casefold()
    kind = answer_type(answer)
    if kind == "date":
        cues = ["kapan", "tahun", "tanggal"] if lang == "id" else ["when", "what year", "which year", "what date"]
        return any(cue in q for cue in cues)
    if kind == "number":
        cues = ["berapa", "berapa banyak", "berapa nilai"] if lang == "id" else ["how many", "how much", "what is the value", "what value"]
        return any(cue in q for cue in cues)
    if kind == "formula":
        cues = ["rumus", "persamaan", "formula", "persamaan matematika"] if lang == "id" else ["formula", "equation"]
        return any(cue in q for cue in cues)
    return None


def generate_question(context, answer, lang, num_beams=4, num_return_sequences=1):
    lang = str(lang).strip().lower()
    context = str(context).strip()
    answer = normalize_space(answer)
    if lang not in {"id", "en"}:
        raise ValueError("lang harus 'id' atau 'en'.")
    if not context or not answer:
        raise ValueError("context dan answer tidak boleh kosong.")
    if num_return_sequences > num_beams:
        raise ValueError("num_return_sequences tidak boleh lebih besar dari num_beams.")

    answer_start = context.find(answer)
    if answer_start < 0:
        answer_start = context.lower().find(answer.lower())
    has_formula, _ = detect_formula_content(context)
    kind = answer_type(answer)

    prompt = make_length_safe_prompt(
        lang,
        context,
        answer,
        kind,
        has_formula,
        answer_start,
    )
    enc = infer_tokenizer(
        prompt,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt",
    ).to(INFER_DEVICE)

    with torch.inference_mode():
        ids = infer_model.generate(
            **enc,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            length_penalty=1.0,
            no_repeat_ngram_size=2,
            max_length=MAX_TARGET_LENGTH,
        )

    return [normalize_space(x) for x in infer_tokenizer.batch_decode(ids, skip_special_tokens=True)]


def show_qg(context, answer, lang, num_beams=4, num_return_sequences=1):
    has_formula, formula_spans = detect_formula_content(context)
    kind = answer_type(answer)
    questions = generate_question(
        context,
        answer,
        lang,
        num_beams=num_beams,
        num_return_sequences=num_return_sequences,
    )

    print("=" * 100)
    print("LANGUAGE       :", lang)
    print("ANSWER TYPE    :", kind)
    print("HAS FORMULA    :", has_formula)
    if formula_spans:
        print("FORMULA SPANS  :", formula_spans[:5])
    print("ANSWER         :", answer)
    print("CONTEXT        :", context)
    for i, question in enumerate(questions, start=1):
        print(f"QUESTION {i:<2}    :", question)
        alignment = answer_type_alignment(question, answer, lang)
        if alignment is not None:
            print(f"TYPE ALIGNMENT : {alignment}")
    print("=" * 100)
    return questions


def interactive_qg():
    print("Mode interaktif. Ketik 'selesai' pada bahasa untuk keluar.")
    while True:
        lang = input("Bahasa [id/en]: ").strip().lower()
        if lang in {"selesai", "exit", "q"}:
            break
        if lang not in {"id", "en"}:
            print("Bahasa harus id atau en.")
            continue
        context = input("Context: ").strip()
        answer = input("Answer: ").strip()
        if not context or not answer:
            print("Context dan answer wajib diisi.")
            continue
        show_qg(context, answer, lang)


# ------------------------- Demonstrasi answer control -------------------------
context_demo_id = (
    "Candi Borobudur adalah candi Buddha di Magelang, Jawa Tengah. "
    "Candi ini dibangun pada abad ke-9 pada masa pemerintahan Dinasti Syailendra "
    "dan ditetapkan sebagai Situs Warisan Dunia UNESCO pada tahun 1991."
)

show_qg(context_demo_id, "Dinasti Syailendra", "id")
show_qg(context_demo_id, "1991", "id")

# ------------------------- Demonstrasi formula -------------------------
formula_demo_id = (
    "Luas persegi panjang dihitung dengan rumus A = p × l. "
    "Jika panjang p = 8 cm dan lebar l = 5 cm, maka luasnya adalah 40 cm²."
)
show_qg(formula_demo_id, "A = p × l", "id")
show_qg(formula_demo_id, "40 cm²", "id")

formula_demo_en = (
    "Ohm's law is expressed as V = I × R. "
    "If current I = 2 A and resistance R = 6 Ω, the voltage is 12 V."
)
show_qg(formula_demo_en, "V = I × R", "en")
show_qg(formula_demo_en, "12 V", "en")

print("\nUntuk pengujian dokumen sendiri, panggil:")
print("interactive_qg()")
